## Calibrate recruitment in the synthetic baseline

Sosna et al. (2019), SI Fig. S5A, shows approximately 60% single-fish events
before the first exposure. We use the complementary 40% recruitment rate
as a visual estimate for this synthetic example.

This calculation fits the chance of at least one secondary escape, averaged
over random starters, thresholds, and dose arrivals on one fixed school.
It does not fit the full cascade-size distribution or reproduce the empirical
network. The response coefficients stay fixed.

Before the first secondary escape, the starter is the only source of doses.
Since `memory_duration` exceeds `active_duration`, every starter dose is retained
throughout its active phase. The total number of packets on each edge is
therefore binomial, with the same arrival probability used by `draw_doses`.


In [1]:
import numpy as np

from network import synthetic_spatial_network


In [2]:
n_fish = 40
dt = 0.001
active_duration = 0.5
memory_duration = 2.0
dose_rate = 1000.0
dose_size = 0.001


In [3]:
## Response coefficients from the pooled first-exposure fit
intercept = 0.06449
distance_coefficient = -3.20552
rank_coefficient = -0.08016


In [4]:
## Synthetic school geometry
arena_size_cm = (66.6, 66.6)
min_separation_cm = 3.0
body_width_cm = 2.0
field_of_view_deg = 360.0
max_visual_distance_cm = None
occlusion = True
spatial_scale = 1.0
network_seed = 2026


In [5]:
adjacency, geometry = synthetic_spatial_network(
    n_fish, intercept, distance_coefficient, rank_coefficient,
    arena_size_cm=arena_size_cm, min_separation_cm=min_separation_cm,
    body_width_cm=body_width_cm, field_of_view_deg=field_of_view_deg,
    max_visual_distance_cm=max_visual_distance_cm, occlusion=occlusion,
    spatial_scale=spatial_scale, random_seed=network_seed, return_geometry=True,
)

nearest_neighbour_cm = geometry["distance_cm"].min(axis=1)
print(f"Median nearest-neighbour distance: {np.median(nearest_neighbour_cm):.2f} cm")


Median nearest-neighbour distance: 6.00 cm


Each row below is an independent trial on this baseline school.
The starter varies between trials; the receiver weights are taken from its
column of `adjacency`. Self-weights are zero, so the starter cannot count
as a secondary escape.


In [6]:
n_calibration_runs = 10000
target_secondary_probability = 0.40
calibration_seed = 2026
calibration_rng = np.random.default_rng([calibration_seed, 3])
active_steps = round(active_duration / dt)


In [7]:
starter_ids = calibration_rng.integers(n_fish, size=n_calibration_runs)
weights = adjacency[:, starter_ids].T
neighbour_count = (adjacency > 0).sum(axis=1)
dose_counts = calibration_rng.binomial(active_steps, dose_rate * weights * dt)
received_evidence = dose_size * dose_counts / neighbour_count


Write each fish's threshold as `mean_threshold * threshold_factor`, with
`threshold_factor` uniform between zero and two, as in the simulation.
A fish responds when `mean_threshold` is below its evidence divided by its
threshold factor. The largest of these ratios determines whether anyone
responds in that trial. Choosing its quantile sets the desired recruitment rate.


In [8]:
threshold_factors = calibration_rng.uniform(0.0, 2.0, received_evidence.shape)
critical_threshold = (received_evidence / threshold_factors).max(axis=1)
calibrated_mean_threshold = np.quantile(critical_threshold, 1 - target_secondary_probability)
calibrated_mean_threshold = round(float(calibrated_mean_threshold), 6)
print(f"Calibrated mean threshold: {calibrated_mean_threshold:.6f}")


Calibrated mean threshold: 0.005596


An independent sample checks the recruitment rate at the selected threshold.


In [9]:
n_validation_runs = 50000
validation_rng = np.random.default_rng([calibration_seed, 4])
validation_starters = validation_rng.integers(n_fish, size=n_validation_runs)
validation_weights = adjacency[:, validation_starters].T
validation_counts = validation_rng.binomial(active_steps, dose_rate * validation_weights * dt)
validation_evidence = dose_size * validation_counts / neighbour_count


In [10]:
validation_thresholds = validation_rng.uniform(
    0.0, 2 * calibrated_mean_threshold, validation_evidence.shape,
)
secondary_escapes = (validation_evidence > validation_thresholds).any(axis=1)
print(f"Runs with secondary escapes: {secondary_escapes.mean():.1%}")


Runs with secondary escapes: 40.0%


The calibrated value is entered explicitly as `mean_threshold` in
[simulation.ipynb](simulation.ipynb). Keep it fixed when varying `spatial_scale`;
recalibrating for each scale would erase the recruitment difference being tested.
A particular starter and threshold draw can still produce only one escape.
